# Regresión lineal en Python: ejemplo completo

Este notebook presenta un flujo reproducible de regresión lineal supervisada utilizando un CSV incluido en el proyecto. El objetivo es predecir `ingresos` a partir de `edad`, `horas_estudio` y `experiencia`.

Se calculan **R², RMSE, MAE, Pearson y Bias**, y se incluyen visualizaciones y análisis de residuos.

## 1. Librerías

Se utiliza `pandas` para manipulación de datos, `numpy` para cálculo numérico, `matplotlib` y `seaborn` para visualización, `scikit-learn` para el modelo y `scipy` para Pearson.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

pd.set_option('display.precision', 4)

## 2. Carga del CSV

El fichero `datos_regresion.csv` está incluido en la misma carpeta que el notebook.

In [ ]:
df = pd.read_csv('datos_regresion.csv')
print(f'Dimensiones: {df.shape}')
display(df.head())

## 3. Exploración inicial

Antes de entrenar el modelo comprobamos estructura, valores ausentes y estadísticos descriptivos.

In [ ]:
print('Información:')
df.info()
print('\nValores ausentes:')
display(df.isnull().sum().to_frame('missing_values'))
print('\nEstadísticas descriptivas:')
display(df.describe())

## 4. Matriz de correlación

La matriz permite observar las relaciones lineales entre las variables.

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df.drop(columns='id').corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Matriz de correlación')
plt.tight_layout()
plt.show()

## 5. Variables predictoras y objetivo

La regresión lineal múltiple se expresa como:

$$\hat{y}=\beta_0+\beta_1x_1+\beta_2x_2+\cdots+\beta_px_p$$

En este ejemplo, `ingresos` es la variable objetivo y `edad`, `horas_estudio` y `experiencia` son las variables predictoras.

In [ ]:
features = ['edad', 'horas_estudio', 'experiencia']
target = 'ingresos'

X = df[features]
y = df[target]

print('Variables predictoras:', features)
print('Variable objetivo:', target)

## 6. División entrenamiento/prueba

Se utiliza el 80 % de las observaciones para entrenamiento y el 20 % para evaluación. `random_state=42` permite reproducir la partición.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f'Tamaño entrenamiento: {len(X_train)}')
print(f'Tamaño prueba: {len(X_test)}')

## 7. Entrenamiento del modelo

`LinearRegression` estima los coeficientes minimizando la suma de errores cuadráticos en el conjunto de entrenamiento.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print(f'Intercepto: {model.intercept_:.4f}')

coeficientes = pd.DataFrame({
    'variable': features,
    'coeficiente': model.coef_
})
display(coeficientes)

## 8. Predicciones

Se predicen los valores del conjunto de prueba, que no fue utilizado para ajustar el modelo.

In [ ]:
y_pred = model.predict(X_test)

resultados = pd.DataFrame({
    'real': y_test.values,
    'predicho': y_pred,
    'error': y_test.values - y_pred
})

display(resultados)

## 9. Métricas de evaluación

**R²** mide la proporción de variabilidad explicada por el modelo.

**RMSE** cuantifica el error cuadrático medio en las mismas unidades de la variable objetivo:

$$RMSE=\sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}$$

**MAE** es el error absoluto medio:

$$MAE=\frac{1}{n}\sum_{i=1}^{n}|y_i-\hat{y}_i|$$

**Pearson** mide la correlación lineal entre valores reales y predichos.

**Bias** se define como el error medio `real - predicho`:

$$Bias=\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)$$

In [ ]:
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
pearson, pearson_pvalue = pearsonr(y_test, y_pred)
bias = np.mean(y_test.to_numpy() - y_pred)

metricas = pd.DataFrame({
    'Metrica': ['R2', 'RMSE', 'MAE', 'Pearson', 'Bias'],
    'Valor': [r2, rmse, mae, pearson, bias]
})

display(metricas)

## 10. Valores reales frente a predicciones

La diagonal representa la situación ideal `real = predicho`. Los puntos próximos a ella corresponden a errores pequeños.

In [ ]:
plt.figure(figsize=(7,6))
plt.scatter(y_test, y_pred, s=70)
lo = min(y_test.min(), y_pred.min())
hi = max(y_test.max(), y_pred.max())
plt.plot([lo, hi], [lo, hi], '--')
plt.xlabel('Valores reales')
plt.ylabel('Valores predichos')
plt.title('Regresión lineal: real vs. predicho')
plt.tight_layout()
plt.show()

## 11. Análisis de residuos

El residuo se define como:

$$e_i=y_i-\hat{y}_i$$

Una distribución de residuos sin patrones sistemáticos es una señal favorable, aunque un diagnóstico completo requiere también analizar residuos frente a predicciones y otros supuestos del modelo.

In [ ]:
residuos = y_test.to_numpy() - y_pred

plt.figure(figsize=(8,5))
sns.histplot(residuos, kde=True)
plt.axvline(0, linestyle='--')
plt.xlabel('Residuo (real - predicho)')
plt.title('Distribución de los residuos')
plt.tight_layout()
plt.show()

## 12. Resumen final

El flujo completo es:

**CSV → exploración → selección de variables → train/test split → regresión lineal → predicción → métricas → análisis de residuos.**

El notebook puede reutilizarse con otro CSV modificando `features` y `target`.

In [ ]:
print('===== RESULTADOS DEL MODELO =====')
print(f'R²      : {r2:.4f}')
print(f'RMSE    : {rmse:.4f}')
print(f'MAE     : {mae:.4f}')
print(f'Pearson : {pearson:.4f}')
print(f'Bias    : {bias:.4f}')